# 03 · Data Validation
Validasi kandidat hasil notebook 02. Tidak ada koreksi otomatis di sini. Kandidat yang gagal harus diperbaiki di notebook 02. Data primer tidak disertakan dalam rekap sintetis.

In [7]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Bisa dijalankan dari root proyek maupun folder notebooks.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'datasets' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder proyek Agentris yang sudah diekstrak.')
RAW = ROOT / 'datasets' / 'raw'
REF = ROOT / 'datasets' / 'reference'
INTERIM = ROOT / 'datasets' / 'interim'
PROCESSED = ROOT / 'datasets' / 'processed'
REPORTS = ROOT / 'reports'
for folder in [INTERIM, PROCESSED, REPORTS / 'figures']:
    folder.mkdir(parents=True, exist_ok=True)
FILES = {
    'penjualan_sintetis': RAW / 'transaksi_penjualan_sintetis_raw_gabungan.csv',
    'pengeluaran_sintetis': RAW / 'transaksi_pengeluaran_sintetis_raw.csv',
    'penjualan_primer': RAW / 'transaksi_penjualan_primer_raw.csv',
    'pengeluaran_primer': RAW / 'transaksi_pengeluaran_primer_raw.csv',
    'menu': REF / 'referensi_menu.csv',
}
def load_raw():
    return {key: pd.read_csv(path, dtype='string') for key, path in FILES.items()}
print('Project root:', ROOT)

Project root: d:\1.KULIAH KUYY\Studi Independent\agentris\data-science


In [10]:
def validate(j, b, ref, raw, reject):
    checks = []
    def check(name, condition):
        checks.append({'pemeriksaan': name, 'lulus': bool(condition)})
    for name, df, required, key in [
        ('Penjualan', j, list(raw['penjualan_sintetis'].columns), 'id_detail'),
        ('Pengeluaran', b, list(raw['pengeluaran_sintetis'].columns), 'no')]:
        if not set(required).issubset(df.columns):
            check(name + ': kolom wajib lengkap', False)
            return pd.DataFrame(checks)
        check(name + ': tidak kosong', len(df) > 0)
        check(name + ': kolom wajib terisi', df[required].notna().all().all() and all(df[c].str.strip().ne('').all() for c in required))
        check(name + ': tidak ada duplikat identik', not df.duplicated().any())
        check(name + ': ID baris unik', df[key].notna().all() and df[key].is_unique)
        dates = pd.to_datetime(df.tanggal, format='%Y-%m-%d', errors='coerce')
        check(name + ': tanggal ISO valid', df.tanggal.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False).all() and dates.notna().all())
        check(name + ': periode Agustus–Oktober 2026', dates.between('2026-08-01','2026-10-31').all())
    nums_j = ['jumlah','harga_satuan','subtotal','total_transaksi']
    check('Penjualan: bilangan bulat positif', all(j[c].str.fullmatch(r'[1-9]\d*', na=False).all() for c in nums_j))
    check('Pengeluaran: nominal bulat positif', b.total.str.fullmatch(r'[1-9]\d*', na=False).all())
    jn = j[nums_j].apply(pd.to_numeric, errors='coerce')
    price = pd.to_numeric(ref.set_index('nama_menu').harga, errors='coerce')
    check('Menu: seluruh nama ada dalam referensi', j.jenis_makanan.isin(price.index).all())
    check('Harga: sesuai referensi', jn.harga_satuan.eq(j.jenis_makanan.map(price)).fillna(False).all())
    check('Subtotal: jumlah × harga', jn.subtotal.eq(jn.jumlah * jn.harga_satuan).fillna(False).all())
    grouped = j.assign(sub=jn.subtotal, grand=jn.total_transaksi).groupby('id_transaksi', dropna=False)
    check('Satu tanggal per transaksi', grouped.tanggal.nunique(dropna=False).eq(1).all())
    check('Satu pembeli per transaksi', grouped.nama_pembeli.nunique(dropna=False).eq(1).all())
    check('Total transaksi konsisten', grouped.grand.nunique(dropna=False).eq(1).all())
    check('Total transaksi = jumlah subtotal', grouped.sub.sum(min_count=1).eq(grouped.grand.first()).fillna(False).all())
    for name, df, key in [('penjualan_sintetis', j, 'id_transaksi'), ('pengeluaran_sintetis', b, 'no')]:
        original = set(raw[name][key].dropna())
        kept = set(df[key].dropna())
        logged = reject[reject.dataset.eq(name)]
        removed = set(logged.id.dropna())
        check(name + ': rekonsiliasi ID raw', kept <= original and removed == original - kept and not (kept & removed))
        check(name + ': alasan karantina tersedia', logged.alasan.notna().all() and logged.alasan.str.strip().ne('').all())
    # ID detail yang hilang/berubah pada transaksi yang dipertahankan perlu ditinjau manual.
    raw_detail = raw['penjualan_sintetis'].loc[raw['penjualan_sintetis'].id_transaksi.isin(j.id_transaksi), ['id_transaksi','id_detail']].dropna().drop_duplicates()
    check('ID detail raw nonkosong tetap terlacak', set(map(tuple,raw_detail.to_numpy())) <= set(map(tuple,j[['id_transaksi','id_detail']].dropna().to_numpy())))
    
    #label sumber data
    for name, df in [
        ('Penjualan', j),
        ('Pengeluaran', b)
    ]:
        check(
            name + ': label sumber data valid',
            'sumber_data' in df.columns
            and df['sumber_data'].eq('sintetis').fillna(False).all()
        )
    return pd.DataFrame(checks)

## Jalankan pemeriksaan
Folder processed hanya ditulis jika semua pemeriksaan otomatis lulus dan review manual telah dikonfirmasi. Review manual mencakup log koreksi, transaksi yang dikarantina, detail tanpa ID, serta dampak data hilang terhadap rekap. Jangan menganggap hasil simulasi sebagai kondisi bisnis aktual.

In [9]:
paths = [INTERIM / f for f in ['penjualan_sintetis_candidate.csv','pengeluaran_sintetis_candidate.csv','reject_log.csv','change_log.csv']]
REVIEW_MANUAL_SELESAI = True  # TODO: konfirmasi sesudah memeriksa log perubahan dan karantina
if not all(p.exists() for p in paths):
    print('Belum ada kandidat lengkap. Selesaikan notebook 02 terlebih dahulu.')
else:
    j, b, reject, changes = [pd.read_csv(p, dtype='string') for p in paths]
    raw = load_raw()
    hasil = validate(j,b,raw['menu'],raw,reject)
    display(hasil)
    display(reject)
    display(changes)
    hasil.to_csv(REPORTS / 'validation_report.csv', index=False)
    if hasil.lulus.all() and REVIEW_MANUAL_SELESAI:
        j.to_csv(PROCESSED / 'penjualan_sintetis_clean.csv', index=False)
        b.to_csv(PROCESSED / 'pengeluaran_sintetis_clean.csv', index=False)
        print('Lulus validasi. Data tersimpan di processed.')
    else:
        print('Belum diekspor. Perbaiki pemeriksaan gagal dan selesaikan review manual. Jika ada ekspor lama, jangan gunakan sebelum validasi ulang.')

,pemeriksaan,lulus
0,Penjualan: tidak kosong,True
1,Penjualan: kolom wajib terisi,True
2,Penjualan: tidak ada duplikat identik,True
3,Penjualan: ID baris unik,True
4,Penjualan: tanggal ISO valid,True
5,Penjualan: periode Agustus–Oktober 2026,True
6,Pengeluaran: tidak kosong,True
7,Pengeluaran: kolom wajib terisi,True
8,Pengeluaran: tidak ada duplikat identik,True
9,Pengeluaran: ID baris unik,True


,dataset,id,alasan
0,penjualan_sintetis,PJ-00044,Rincian ID/menu hilang atau menu tidak dikenali
1,penjualan_sintetis,PJ-00048,Rincian ID/menu hilang atau menu tidak dikenali
2,penjualan_sintetis,PJ-00091,Rincian ID/menu hilang atau menu tidak dikenali
3,penjualan_sintetis,PJ-00148,Tanggal tidak dapat dipulihkan atau konflik
4,penjualan_sintetis,PJ-00176,Tanggal tidak dapat dipulihkan atau konflik
5,penjualan_sintetis,PJ-00192,Tanggal tidak dapat dipulihkan atau konflik
6,penjualan_sintetis,PJ-00203,Rincian ID/menu hilang atau menu tidak dikenali
7,penjualan_sintetis,PJ-00214,Rincian ID/menu hilang atau menu tidak dikenali
8,penjualan_sintetis,PJ-00312,Rincian ID/menu hilang atau menu tidak dikenali
9,penjualan_sintetis,PJ-00337,Rincian ID/menu hilang atau menu tidak dikenali


,dataset,id,kolom,sebelum,sesudah,alasan
0,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",<NA>,Duplikat identik; satu salinan dipertahankan
1,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",<NA>,Duplikat identik; satu salinan dipertahankan
2,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",<NA>,Duplikat identik; satu salinan dipertahankan
3,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",<NA>,Duplikat identik; satu salinan dipertahankan
4,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",<NA>,Duplikat identik; satu salinan dipertahankan
...,...,...,...,...,...,...
458,pengeluaran_primer,AG-011,total,"24,5",24500,Normalisasi Rupiah; skala ribuan asumsi sumber
459,pengeluaran_primer,AG-012,total,28,28000,Normalisasi Rupiah; skala ribuan asumsi sumber
460,pengeluaran_primer,AG-013,total,20,20000,Normalisasi Rupiah; skala ribuan asumsi sumber
461,penjualan_primer,AS-001,nama_pembeli,Shoopee,Shopee,Standardisasi ejaan kanal; jenis layanan tidak...


Lulus validasi. Data tersimpan di processed.
